<h3>Modelling

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from evaluate_binary_classifier import *

In [2]:
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
INTERIM_DATA_DIR = DATA_DIR / "interim"

In [3]:
X_test = pd.read_pickle(INTERIM_DATA_DIR / "X_test_processed.pkl")
X_train = pd.read_pickle(INTERIM_DATA_DIR / "X_train_sampled.pkl")
y_test = pd.read_pickle(INTERIM_DATA_DIR / "y_test_processed.pkl")
y_train = pd.read_pickle(INTERIM_DATA_DIR / "y_train_sampled.pkl")

Here we will try several different binary classification models.

<h4>Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV

In [8]:
# Base model:

lr_model = LogisticRegression()

lr_model.fit(X_train,y_train)

y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)[:,1]

lr_model_base = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8913
Balanced Accuracy: 0.7866
Precision: 0.3656
Recall: 0.6639
F1 Score: 0.4715
Matthews Corrcoef: 0.4397
ROC AUC: 0.8874
Log Loss: 0.3239

Confusion Matrix:
[[9715  970]
 [ 283  559]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     10685
           1       0.37      0.66      0.47       842

    accuracy                           0.89     11527
   macro avg       0.67      0.79      0.71     11527
weighted avg       0.93      0.89      0.91     11527



In [15]:
# Hyperparameter tuning and K-fold validation (fixed)

lr_model = LogisticRegression(max_iter=1000, random_state=13)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=13
)

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"],
    "class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=lr_model,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_search.fit(X_train, y_train)

# Training score
print("Best CV score:", grid_search.best_score_)
print("Best parameters:")
print(grid_search.best_params_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best CV score: 0.8887995024006449
Best parameters:
{'C': 1, 'class_weight': None, 'penalty': 'l2', 'solver': 'lbfgs'}


c:\Users\cochr\Data Science Projects\lA82lBNDpo9AlAnK\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [13]:
# Test score:
best_lr = grid_search.best_estimator_
y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:,1]

best_lr_evaluate = evaluate_binary_classifier(y_test=y_test,
                                           y_pred=y_pred,
                                           y_proba=y_proba)


===== Binary Classification Evaluation =====

Accuracy: 0.8913
Balanced Accuracy: 0.7866
Precision: 0.3656
Recall: 0.6639
F1 Score: 0.4715
Matthews Corrcoef: 0.4397
ROC AUC: 0.8874
Log Loss: 0.3239

Confusion Matrix:
[[9715  970]
 [ 283  559]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     10685
           1       0.37      0.66      0.47       842

    accuracy                           0.89     11527
   macro avg       0.67      0.79      0.71     11527
weighted avg       0.93      0.89      0.91     11527



<h4>Decision Trees